# dim_indicators: Gold

Builds `pondolens.dev_gold.dim_indicator`. `FACT_REGIONAL_INDICATOR` uses this dimension to look up `indicator_id`.

| Indicator | Silver source | Measure type | Unit |
|---|---|---|---|
| `unemployment` | `pondolens.dev_silver.unemployment_rate` | `unemployment_rate` | `% of labor force` |
| `grdp` | `pondolens.dev_silver.grdp_per_capita` | `grdp_per_capita`, separated by `price_basis` | `pesos` |
| `service_access` | `pondolens.dev_silver.service_access_available` | From `service_measure` | As recorded in Silver |

## Rules

- **Business key:** `indicator`, `measure_type`, `price_basis`, `unit`, `service`, `breakdown_type`, and `breakdown_value`. Each unique combination represents one indicator.
- **Indicator IDs:** Start at `40001` and must not exceed `49999`. Existing indicators retain their IDs; new combinations receive IDs after the current maximum.
- **Not applicable:** Use `'N/A'` for attributes that do not apply.
- - **GRDP price basis:** Include only `price_basis = '2018'` from Silver, representing constant 2018 prices. 
- **Unemployment and GRDP:** Use `Total` for `breakdown_type` and `All` for `breakdown_value`.
- **Service-access source:** Read from `pondolens.dev_silver.service_access_available`.
- **Excluded breakdowns:** Exclude Rural and Urban from staging. The filter uses `UPPER()` and `TRIM()` so capitalization and surrounding spaces do not affect matching.
- **Service-access units:** Retain the source units, with blank or null values replaced by `'N/A'`.

## Reruns and new data

- Rerunning with unchanged source data inserts no additional rows.
- New indicator combinations are inserted automatically.
- Changes to business-key values create new combinations rather than updating existing indicators.
- Indicators already in Gold are retained, even if they disappear from staging. This includes previously loaded Rural and Urban indicators.
- The merge stops if a new ID would exceed `49999`.

## Execution and checks

- Run all cells from top to bottom.
- The table uses Delta `Serializable` isolation to detect conflicting concurrent writes. Rerun if a concurrency conflict occurs.
- Checks validate required values, GRDP price basis, ID range, duplicate IDs, duplicate business keys, and coverage of staged indicators in Gold.

## 1. Create the Gold table

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pondolens.dev_gold;

CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_indicator (
  indicator_id INT NOT NULL COMMENT 'Surrogate key, 40001-49999',
  indicator STRING NOT NULL COMMENT 'unemployment, grdp, service_access',
  measure_type STRING NOT NULL COMMENT 'unemployment_rate, grdp_per_capita, electricity_access, ...',
  price_basis STRING NOT NULL COMMENT '2018 = constant 2018 prices; 2021-2025 = current prices; N/A otherwise',
  unit STRING NOT NULL COMMENT '% of labor force, pesos, % of population, % of households, ...',
  service STRING NOT NULL COMMENT 'Water, Electricity, Internet; N/A otherwise',
  breakdown_type STRING NOT NULL,
  breakdown_value STRING NOT NULL,
  CONSTRAINT dim_indicator_pk PRIMARY KEY (indicator_id)
)
USING DELTA
TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')
COMMENT 'Gold indicator dimension. Business key: indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value.';

-- Makes sure a table created by an earlier version also uses Serializable isolation.
ALTER TABLE pondolens.dev_gold.dim_indicator SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable');

## 2. Build and validate the indicator list

- Select distinct indicator combinations from the Silver sources.
- Include GRDP indicators only with a price basis of `2018`.
- Read service-access indicators from `pondolens.dev_silver.service_access_available`.
- Exclude Rural and Urban breakdowns using `UPPER()` and `TRIM()` when filtering.
- Check required values, GRDP price basis, and existing Gold IDs and business keys.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_indicator_stage AS
SELECT DISTINCT
  'unemployment' AS indicator, 'unemployment_rate' AS measure_type, 'N/A' AS price_basis,
  '% of labor force' AS unit, 'N/A' AS service, 'Total' AS breakdown_type, 'All' AS breakdown_value
FROM pondolens.dev_silver.unemployment_rate
UNION
SELECT DISTINCT
  'grdp', 'grdp_per_capita', CAST(price_basis AS STRING),
  'pesos', 'N/A', 'Total', 'All'
FROM pondolens.dev_silver.grdp_per_capita
WHERE CAST(price_basis AS STRING) = '2018'
UNION
SELECT DISTINCT
  'service_access',
  LOWER(TRIM(BOTH '_' FROM REGEXP_REPLACE(TRIM(service_measure), '[^A-Za-z0-9]+', '_'))),  -- 'Electricity access' -> 'electricity_access'
  'N/A',
  COALESCE(NULLIF(TRIM(unit), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(service), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(breakdown_type), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(breakdown_value), ''), 'N/A')
FROM pondolens.dev_silver.service_access_available
WHERE UPPER(TRIM(COALESCE(breakdown_value, '')))
      NOT IN ('RURAL', 'URBAN');

SELECT
  ASSERT_TRUE(COUNT(*) > 0, 'No indicators found in Silver.') AS has_rows,
  ASSERT_TRUE(COUNT_IF(NULLIF(measure_type, '') IS NULL OR NULLIF(price_basis, '') IS NULL) = 0,
    'Blank measure_type or price_basis. Check the Silver source.') AS keys_present,
  ASSERT_TRUE(COUNT_IF(indicator = 'grdp' AND price_basis != '2018') = 0,
    'GRDP price_basis must be 2018.') AS valid_price_basis
FROM dim_indicator_stage;

-- Existing Gold must already have unique IDs, unique keys and IDs in range.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT indicator_id)
  AND COUNT(*) = COUNT(DISTINCT STRUCT(indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value))
  AND COUNT_IF(indicator_id NOT BETWEEN 40001 AND 49999) = 0,
  'Gold has duplicate IDs, duplicate keys or IDs outside 40001-49999.'
)
FROM pondolens.dev_gold.dim_indicator;

SELECT *
FROM dim_indicator_stage
ORDER BY indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value;

## 3. Merge new indicators (IDs assigned inside the MERGE)

- `ROW_NUMBER()` runs only on indicators not yet in Gold, continuing from the current maximum (40000 when empty).
- Every column except `indicator_id` is part of the business key, so matched rows have nothing to update: the MERGE only inserts.
- An ID above 49999 stops the MERGE before anything is written.

In [0]:
%sql
MERGE INTO pondolens.dev_gold.dim_indicator AS t
USING (
  SELECT
    n.*,
    CASE WHEN n.new_id <= 49999 THEN CAST(n.new_id AS INT)
         ELSE CAST(RAISE_ERROR('indicator_id would exceed 49999 (FACT_INDICATORS range starts at 50000).') AS INT)
    END AS indicator_id
  FROM (
    SELECT s.*, l.last_id + ROW_NUMBER() OVER (ORDER BY s.indicator, s.measure_type, s.price_basis, s.unit, s.service, s.breakdown_type, s.breakdown_value) AS new_id
    FROM dim_indicator_stage AS s
    LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS d
      ON s.indicator = d.indicator
      AND s.measure_type = d.measure_type
      AND s.price_basis = d.price_basis
      AND s.unit = d.unit
      AND s.service = d.service
      AND s.breakdown_type = d.breakdown_type
      AND s.breakdown_value = d.breakdown_value
    CROSS JOIN (SELECT COALESCE(MAX(indicator_id), 40000) AS last_id FROM pondolens.dev_gold.dim_indicator) AS l
  ) AS n
) AS s
ON t.indicator = s.indicator
  AND t.measure_type = s.measure_type
  AND t.price_basis = s.price_basis
  AND t.unit = s.unit
  AND t.service = s.service
  AND t.breakdown_type = s.breakdown_type
  AND t.breakdown_value = s.breakdown_value
WHEN NOT MATCHED THEN INSERT (indicator_id, indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value)
VALUES (s.indicator_id, s.indicator, s.measure_type, s.price_basis, s.unit, s.service, s.breakdown_type, s.breakdown_value);

## 4. Post-merge checks and results

Final safety net: IDs and business keys are unique, and every Silver indicator is in Gold.

In [0]:
%sql
SELECT
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT indicator_id) AND COUNT(*) = COUNT(DISTINCT STRUCT(indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value)),
    'Duplicate IDs or business keys after the merge.') AS unique_ids_and_keys
FROM pondolens.dev_gold.dim_indicator;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Some Silver indicators are missing from Gold.')
FROM dim_indicator_stage AS s
LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS t
  ON s.indicator = t.indicator
  AND s.measure_type = t.measure_type
  AND s.price_basis = t.price_basis
  AND s.unit = t.unit
  AND s.service = t.service
  AND s.breakdown_type = t.breakdown_type
  AND s.breakdown_value = t.breakdown_value;

-- Indicators no longer in Silver (kept on purpose; normally empty).
SELECT t.*
FROM pondolens.dev_gold.dim_indicator AS t
LEFT ANTI JOIN dim_indicator_stage AS s
  ON t.indicator = s.indicator
  AND t.measure_type = s.measure_type
  AND t.price_basis = s.price_basis
  AND t.unit = s.unit
  AND t.service = s.service
  AND t.breakdown_type = s.breakdown_type
  AND t.breakdown_value = s.breakdown_value;

SELECT * FROM pondolens.dev_gold.dim_indicator ORDER BY indicator_id;

   
## 5. Data quality and idempotency checks

- **DQ 1–2:** Duplicate `indicator_id` and exact duplicate rows.
- **DQ 3:** Duplicate business keys (7 key columns, ignoring the surrogate ID).
- **DQ 4:** All IDs within 40001–49999.
- **DQ 5:** Idempotency — snapshot the table, re-run the MERGE, and verify the row count and ID sum are unchanged.
- **DQ 6:** New-batch coverage — every staged indicator exists in Gold.

Each query returns `fail_count = 0` when the check passes.

In [0]:
%sql
WITH gold_checks AS (
  SELECT
    *,
    COUNT(*) OVER (
      PARTITION BY indicator_id
    ) AS id_occurrences,
    COUNT(*) OVER (
      PARTITION BY
        indicator_id, indicator, measure_type, price_basis,
        unit, service, breakdown_type, breakdown_value
    ) AS row_occurrences,
    COUNT(*) OVER (
      PARTITION BY
        indicator, measure_type, price_basis,
        unit, service, breakdown_type, breakdown_value
    ) AS key_occurrences
  FROM pondolens.dev_gold.dim_indicator
),

missing_from_gold AS (
  SELECT s.*
  FROM dim_indicator_stage AS s
  LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS d
    ON s.indicator = d.indicator
    AND s.measure_type = d.measure_type
    AND s.price_basis = d.price_basis
    AND s.unit = d.unit
    AND s.service = d.service
    AND s.breakdown_type = d.breakdown_type
    AND s.breakdown_value = d.breakdown_value
),

results AS (
  SELECT
    1 AS check_order,
    'Duplicate indicator IDs' AS check_name,
    COUNT(*) AS rows_inspected,
    COUNT_IF(id_occurrences > 1) AS affected_rows
  FROM gold_checks

  UNION ALL

  SELECT
    2,
    'Exact duplicate rows',
    COUNT(*),
    COUNT_IF(row_occurrences > 1)
  FROM gold_checks

  UNION ALL

  SELECT
    3,
    'Duplicate business keys',
    COUNT(*),
    COUNT_IF(key_occurrences > 1)
  FROM gold_checks

  UNION ALL

  SELECT
    4,
    'Missing or out-of-range indicator IDs',
    COUNT(*),
    COUNT_IF(
      indicator_id IS NULL
      OR indicator_id NOT BETWEEN 40001 AND 49999
    )
  FROM gold_checks

  UNION ALL

  SELECT
    5,
    'Staged indicators missing from Gold / rows a rerun would insert',
    (SELECT COUNT(*) FROM dim_indicator_stage),
    COUNT(*)
  FROM missing_from_gold

  UNION ALL

  SELECT
    6,
    'Rural or Urban indicators in staging',
    COUNT(*),
    COUNT_IF(
      UPPER(TRIM(breakdown_value)) IN ('RURAL', 'URBAN')
    )
  FROM dim_indicator_stage

  UNION ALL

  SELECT
    7,
    'Rural or Urban indicators retained in Gold',
    COUNT(*),
    COUNT_IF(
      UPPER(TRIM(breakdown_value)) IN ('RURAL', 'URBAN')
    )
  FROM gold_checks
)

SELECT
  check_name AS `check`,
  rows_inspected,
  affected_rows,
  ROUND(
    100.0 * affected_rows / NULLIF(rows_inspected, 0),
    2
  ) AS affected_percentage
FROM results
ORDER BY check_order;

## For FACT_REGIONAL_INDICATOR (lookup example)

```sql
-- Unemployment
SELECT d.indicator_id, u.year, u.region_code, u.unemployment_rate AS measurement
FROM pondolens.dev_silver.unemployment_rate AS u
JOIN pondolens.dev_gold.dim_indicator AS d ON d.indicator = 'unemployment';

-- GRDP per capita: 2018 price basis only
SELECT
  d.indicator_id,
  g.year,
  g.region_code,
  g.grdp_per_capita AS measurement
FROM pondolens.dev_silver.grdp_per_capita AS g
JOIN pondolens.dev_gold.dim_indicator AS d
  ON d.indicator = 'grdp'
  AND d.price_basis = CAST(g.price_basis AS STRING)
WHERE CAST(g.price_basis AS STRING) = '2018';
```

For service access, apply the same `measure_type`, `unit`, `service` and breakdown cleaning as in step 2, then join on all seven key columns.